# ContractRisk — prueba visual 03 de extremo a extremo

GPU T4, sesión nueva. Diez casos con revisión humana registrada; referencias separadas de la inferencia. Ejecutar en orden. No cambia el Space ni main.

Carga VLM y generador juntos y realiza diez lecturas nuevas, una por solicitud. Una solicitud histórica de calentamiento queda excluida. Se registra tiempo de carga por separado, latencia sincronizada y memoria CUDA. No hay cuantización ni fallback automático: si no cabe, se conserva el error. No es todavía el scorecard final ni una comparación de eficiencia.

Antes de ejecutar, habilitar el acceso de este notebook al secreto `WANDB_API_KEY` en el panel de secretos. No introducir claves en código ni salidas. La prueba valida W&B antes de cargar modelos; los eventos no incluyen preguntas, OCR ni imágenes.


In [ ]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_visual03_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

from google.colab import userdata
try:
    key = userdata.get('WANDB_API_KEY')
    if not key:
        raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY'] = key
    del key
except Exception:
    raise RuntimeError('Habilita el acceso de ESTE notebook a WANDB_API_KEY en el panel de secretos de Colab. No pegues ni imprimas la clave.') from None
print('Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.')


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-integration.txt'], 'installation.log')
import torch
run_visible([sys.executable, '-c', 'from final.run_integration import fixture_font_path; print(fixture_font_path().name)'], 'font_check.log')
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
RESULTS = ROOT / 'visual03_results'
run_visible([sys.executable, '-u', 'final/visual_probe.py', '--output', str(RESULTS), '--require-wandb'], 'visual03.log')


## Diagnóstico posterior, separado de la inferencia

El proceso siguiente carga las referencias después de generar respuestas. Mide coincidencia de campos, presencia literal de modalidad/valor, citas y abstención. No es juez semántico ni cumplimiento de dominio final. Todas las respuestas requieren auditoría.


In [ ]:
if (RESULTS / 'predictions.jsonl').exists():
    run_visible([sys.executable, '-u', 'final/score_visual_probe.py', '--results', str(RESULTS)], 'visual03_diagnostics.log')
else:
    print('No se generaron predicciones: revisar metadata y visual03.log; conservar la evidencia del fallo.')


## Inspección y descarga de evidencia

Estas celdas también permiten conservar fallos. El ZIP incluye respuestas, trazas, metadata y el paquete visual revisado. Las latencias de guard y las de inferencia se identifican por caso; no atribuir una abstención determinista al VLM.


In [ ]:
for name in ['metadata.json', 'diagnostics_v1.json']:
    path = RESULTS / name
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        print(name, json.dumps(data.get('summary', data), ensure_ascii=False, indent=2))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
reviewed = RESULTS / 'reviewed_visual_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/visual_candidates_v1', reviewed)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_visual03_resultados'), 'zip', RESULTS)
files.download(package)
